In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

BRONZE_TABLE = "weather.bronze.ana_rio_uruguai"
DAILY_PATH = "/Volumes/weather/raw/ana_volume/json/"

# Infer schema from daily JSON files
daily_df = (
    spark.read
         .option("multiLine", True)
         .json(DAILY_PATH)
)

# Ajustar el schema de bronze según daily_df
bronze_schema = daily_df.schema

# Leer daily con schema ajustado
daily_df = (
    spark.read
         .schema(bronze_schema)
         .option("multiLine", True)
         .json(DAILY_PATH)
)

# Opcional pero recomendable: dedupe interno del batch
daily_df = daily_df.dropDuplicates(["codigoestacao", "Data_Hora_Medicao"])

delta_table = DeltaTable.forName(spark, BRONZE_TABLE)

(
    delta_table.alias("t")
    .merge(
        daily_df.alias("s"),
        "t.codigoestacao = s.codigoestacao AND t.Data_Hora_Medicao = s.Data_Hora_Medicao"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
%sql
select count(*) from weather.bronze.ana_rio_uruguai